# Colab 训练（DNTR）




## 1. 检查运行环境与依赖

检查 Python、PyTorch、CUDA 与关键包状态；缺失时自动安装。

In [1]:
import importlib
import platform
import subprocess
import sys

print('Python:', sys.version)
print('Platform:', platform.platform())


def run_cmd(cmd, timeout=1200):
    """Run shell command with timeout to avoid hanging forever."""
    print('\n[RUN]', ' '.join(cmd))
    try:
        subprocess.run(cmd, check=True, timeout=timeout)
        return True
    except subprocess.TimeoutExpired:
        print('[TIMEOUT] command exceeded timeout:', timeout, 'seconds')
        return False
    except subprocess.CalledProcessError as e:
        print('[FAILED] return code:', e.returncode)
        return False


def get_ver(module_name):
    try:
        mod = importlib.import_module(module_name)
        return getattr(mod, '__version__', 'unknown')
    except Exception:
        return None


# Show current env first
for pkg, mod in {
    'torch': 'torch',
    'torchvision': 'torchvision',
    'mmengine': 'mmengine',
    'mmdet': 'mmdet',
    'mmcv': 'mmcv',
    'pycocotools': 'pycocotools',
    'timm': 'timm',
    'torchprofile': 'torchprofile',
}.items():
    ver = get_ver(mod)
    print(f'{pkg}:', ver if ver else 'NOT INSTALLED')

# Step 1: install lightweight deps first
run_cmd([sys.executable, '-m', 'pip', 'install', '-U',
         'mmengine', 'mmdet', 'torchprofile', 'pycocotools', 'timm'], timeout=900)

# Step 2: install openmim then mmcv (prefer prebuilt wheel)
run_cmd([sys.executable, '-m', 'pip', 'install', '-U', 'openmim'], timeout=300)
ok_mmcv = run_cmd([sys.executable, '-m', 'mim', 'install', 'mmcv'], timeout=1200)

print('\n===== Final Package Versions =====')
for pkg, mod in {
    'torch': 'torch',
    'torchvision': 'torchvision',
    'mmengine': 'mmengine',
    'mmdet': 'mmdet',
    'mmcv': 'mmcv',
    'pycocotools': 'pycocotools',
    'timm': 'timm',
    'torchprofile': 'torchprofile',
}.items():
    ver = get_ver(mod)
    print(f'{pkg}:', ver if ver else 'NOT INSTALLED')

if not ok_mmcv:
    print('\n[IMPORTANT] mmcv install failed or timed out.')
    print('This usually means your current Colab runtime (Python/Torch/CUDA combo) has no matching prebuilt mmcv wheel.')
    print('Use one of these options:')
    print('1) Switch to fallback/older Colab runtime and rerun this cell.')
    print('2) Use Kaggle training environment where mmcv wheel is easier to match.')
    print('3) Build mmcv from source (slow, often >30 min, not recommended on free Colab).')

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
mmengine: NOT INSTALLED
mmdet: NOT INSTALLED
mmcv: NOT INSTALLED
pycocotools: unknown
timm: 1.0.25
torchprofile: NOT INSTALLED

[RUN] /usr/bin/python3 -m pip install -U mmengine mmdet torchprofile pycocotools timm

[RUN] /usr/bin/python3 -m pip install -U openmim

[RUN] /usr/bin/python3 -m mim install mmcv
[FAILED] return code: 1

===== Final Package Versions =====
torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
mmengine: 0.10.7
mmdet: NOT INSTALLED
mmcv: NOT INSTALLED
pycocotools: unknown
timm: 1.0.25
torchprofile: 0.1.0

[IMPORTANT] mmcv install failed or timed out.
This usually means your current Colab runtime (Python/Torch/CUDA combo) has no matching prebuilt mmcv wheel.
Use one of these options:
1) Switch to fallback/older Colab runtime and rerun this cell.
2) Use Kaggle training environment where mmcv wheel is easier to match.
3) Bu

In [1]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Selected device:', device)

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'Total GPU Memory: {total_mem:.2f} GB')
else:
    print('No CUDA GPU detected. Running on CPU.')

# TPU 检测（Colab TPU 场景，可选）
try:
    import torch_xla.core.xla_model as xm
    print('TPU detected:', xm.xla_device())
except Exception:
    print('TPU not active in current runtime.')

Selected device: cuda
GPU: Tesla T4
Total GPU Memory: 14.56 GB
TPU not active in current runtime.


## 3. 验证 GPU/TPU 与设备信息

输出设备状态并自动选择训练设备。

In [5]:
import random
import numpy as np
import torch

# 训练超参数
learning_rate = 1e-3
batch_size = 4
epochs = 3
seed = 42

# 路径配置（可按需修改）
PROJECT_DIR = '/content/drive/MyDrive/dntr_mmdet3_migration'
DATA_ROOT = '/content/drive/MyDrive/AI-TOD'
WORK_DIR = '/content/drive/MyDrive/work_dirs/aitod_DNTR_mask_mmdet3_train'

# 复现性设置
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print('learning_rate=', learning_rate)
print('batch_size=', batch_size)
print('epochs=', epochs)
print('seed=', seed)
print('PROJECT_DIR=', PROJECT_DIR)
print('DATA_ROOT=', DATA_ROOT)
print('WORK_DIR=', WORK_DIR)

learning_rate= 0.001
batch_size= 4
epochs= 3
seed= 42
PROJECT_DIR= /content/drive/MyDrive/dntr_mmdet3_migration
DATA_ROOT= /content/drive/MyDrive/AI-TOD
WORK_DIR= /content/drive/MyDrive/work_dirs/aitod_DNTR_mask_mmdet3_train


## 2. 创建 `.ipynb` 并配置训练参数

在这里统一定义超参数与路径，保证训练可复现。

## 4. 编写最小可运行训练循环

这里使用一个轻量 MLP + 随机数据集，快速验证训练与验证流程是否正常。

In [6]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# 伪造一个二分类数据集
num_features = 32
num_train = 2048
num_val = 512

X_train = torch.randn(num_train, num_features)
y_train = (X_train.sum(dim=1) > 0).long()
X_val = torch.randn(num_val, num_features)
y_val = (X_val.sum(dim=1) > 0).long()

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size, shuffle=False)

model = nn.Sequential(
    nn.Linear(num_features, 128),
    nn.ReLU(),
    nn.Linear(128, 2),
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

history = []

for epoch in range(1, epochs + 1):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for xb, yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)
        pred = logits.argmax(dim=1)
        train_correct += (pred == yb).sum().item()
        train_total += xb.size(0)

    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            logits = model(xb)
            loss = criterion(logits, yb)
            val_loss += loss.item() * xb.size(0)
            pred = logits.argmax(dim=1)
            val_correct += (pred == yb).sum().item()
            val_total += xb.size(0)

    epoch_log = {
        'epoch': epoch,
        'train_loss': train_loss / train_total,
        'train_acc': train_correct / train_total,
        'val_loss': val_loss / val_total,
        'val_acc': val_correct / val_total,
    }
    history.append(epoch_log)
    print(epoch_log)

{'epoch': 1, 'train_loss': 0.30727799038277226, 'train_acc': 0.90283203125, 'val_loss': 0.1442926424278994, 'val_acc': 0.96484375}
{'epoch': 2, 'train_loss': 0.10614714577118889, 'train_acc': 0.974609375, 'val_loss': 0.09524922291382154, 'val_acc': 0.97265625}
{'epoch': 3, 'train_loss': 0.06310317149307787, 'train_acc': 0.98681640625, 'val_loss': 0.08167979902403033, 'val_acc': 0.974609375}


## 5. 保存模型与训练日志

保存 checkpoint、最佳模型以及 CSV/JSON 训练日志。

In [7]:
import csv
import json
from pathlib import Path

save_dir = Path('./notebook_outputs')
save_dir.mkdir(parents=True, exist_ok=True)

# 保存最后一个 checkpoint
last_ckpt = save_dir / 'last_checkpoint.pt'
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'history': history,
    },
    last_ckpt,
)

# 保存最佳模型（按 val_acc）
best = max(history, key=lambda x: x['val_acc']) if history else None
best_model_path = save_dir / 'best_model.pt'
torch.save(model.state_dict(), best_model_path)

# 保存 JSON 日志
json_path = save_dir / 'train_log.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(history, f, ensure_ascii=False, indent=2)

# 保存 CSV 日志
csv_path = save_dir / 'train_log.csv'
if history:
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=history[0].keys())
        writer.writeheader()
        writer.writerows(history)

print('Saved:', last_ckpt)
print('Saved:', best_model_path)
print('Saved:', json_path)
print('Saved:', csv_path)
print('Best epoch:', best['epoch'] if best else None)
print('Best val_acc:', best['val_acc'] if best else None)

Saved: notebook_outputs/last_checkpoint.pt
Saved: notebook_outputs/best_model.pt
Saved: notebook_outputs/train_log.json
Saved: notebook_outputs/train_log.csv
Best epoch: 3
Best val_acc: 0.974609375


## DNTR 实战（Colab）

下面单元用于你当前项目的实际训练。

In [3]:
# Colab 挂载 Google Drive（在 Colab 里执行）
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Drive mounted')
except Exception as e:
    print('Not in Colab or mount skipped:', e)

Mounted at /content/drive
Drive mounted


In [14]:
import os
import sys
from pathlib import Path


def find_first_existing(paths):
    for path in paths:
        if Path(path).exists():
            return str(Path(path))
    return None


# Kaggle / Colab 双环境兼容
project_candidates = [
    '/kaggle/working/dntr_mmdet3_migration',
    '/kaggle/input/dntr-mmdet3-migration/dntr_mmdet3_migration',
    '/kaggle/input/dntr-mmdet3-migration',
    '/content/drive/MyDrive/dntr_mmdet3_migration',
]

data_candidates = [
    '/kaggle/input/ai-tod/AI-TOD',
    '/kaggle/input/ai-tod',
    '/content/drive/MyDrive/AI-TOD',
]

PROJECT_DIR = find_first_existing(project_candidates)
DATA_ROOT = find_first_existing(data_candidates)

if PROJECT_DIR is None:
    raise FileNotFoundError(
        '未找到项目目录。请检查以下候选路径是否存在:\n' + '\n'.join(project_candidates)
    )
if DATA_ROOT is None:
    raise FileNotFoundError(
        '未找到数据集目录。请检查以下候选路径是否存在:\n' + '\n'.join(data_candidates)
    )

CFG_PATH = f'{PROJECT_DIR}/configs/aitod_DNTR_mask_mmdet3_train.py'

# Kaggle 输出建议放 working，Colab 放 Drive
if Path('/kaggle/working').exists():
    WORK_DIR = '/kaggle/working/work_dirs/aitod_DNTR_mask_mmdet3_train'
else:
    WORK_DIR = '/content/drive/MyDrive/work_dirs/aitod_DNTR_mask_mmdet3_train'

if not Path(CFG_PATH).exists():
    raise FileNotFoundError(f'CFG_PATH 不存在: {CFG_PATH}')

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

existing_pp = os.environ.get('PYTHONPATH', '')
os.environ['PYTHONPATH'] = (PROJECT_DIR + ':' + existing_pp).rstrip(':')

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)

cfg_text = Path(CFG_PATH).read_text(encoding='utf-8')
cfg_text = cfg_text.replace("data_root = 'D:/DNTR/AI-TOD/'", f"data_root = '{DATA_ROOT}/'")
cfg_text = cfg_text.replace("work_dir = './work_dirs/aitod_DNTR_mask_mmdet3_train'", f"work_dir = '{WORK_DIR}'")
Path(CFG_PATH).write_text(cfg_text, encoding='utf-8')

print('✓ Config updated:', CFG_PATH)
print('  PROJECT_DIR =', PROJECT_DIR)
print('  DATA_ROOT   =', DATA_ROOT)
print('  WORK_DIR    =', WORK_DIR)
print('  PYTHONPATH  =', os.environ['PYTHONPATH'])
print('\nDirectory check:')
print('  project exists:', Path(PROJECT_DIR).exists())
print('  data exists   :', Path(DATA_ROOT).exists())
print('  config exists :', Path(CFG_PATH).exists())

✓ Config updated: /content/drive/MyDrive/dntr_mmdet3_migration/configs/aitod_DNTR_mask_mmdet3_train.py
  PROJECT_DIR = /content/drive/MyDrive/dntr_mmdet3_migration
  DATA_ROOT   = /content/drive/MyDrive/AI-TOD
  WORK_DIR    = /content/drive/MyDrive/work_dirs/aitod_DNTR_mask_mmdet3_train
  PYTHONPATH  = /content/drive/MyDrive/dntr_mmdet3_migration:/content/drive/MyDrive/dntr_mmdet3_migration:/env/python

Directory check:
  project exists: True
  data exists   : True
  config exists : True


In [5]:
import os

train_ann = os.path.join(DATA_ROOT, 'annotations', 'instances_train.json')
val_ann = os.path.join(DATA_ROOT, 'annotations', 'instances_val.json')
print('train annotation exists:', os.path.exists(train_ann), train_ann)
print('val annotation exists:', os.path.exists(val_ann), val_ann)

train annotation exists: False /content/drive/MyDrive/AI-TOD/annotations/instances_train.json
val annotation exists: False /content/drive/MyDrive/AI-TOD/annotations/instances_val.json


In [13]:
# 训练前依赖自检（当前 Colab 内核）
import importlib
import subprocess
import sys


def _has_mod(name: str) -> bool:
    try:
        importlib.import_module(name)
        return True
    except Exception:
        return False


def _mod_ver(name: str) -> str:
    try:
        mod = importlib.import_module(name)
        return getattr(mod, '__version__', 'unknown')
    except Exception:
        return 'MISSING'


print('Python:', sys.version)
try:
    import torch
    print('torch:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
    print('torch cuda:', torch.version.cuda)
except Exception as e:
    print('torch inspect failed:', e)

need_base = [
    ('mmengine', 'mmengine'),
    ('pycocotools', 'pycocotools'),
    ('timm', 'timm'),
    ('torchprofile', 'torchprofile'),
]

missing_base = [pip_name for mod_name, pip_name in need_base if not _has_mod(mod_name)]
if missing_base:
    print('Installing missing base packages:', missing_base)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', *missing_base], check=False)
else:
    print('Base packages already installed.')

# mmdet 依赖 mmcv，顺序上先处理 mmcv 再处理 mmdet
if not _has_mod('mmcv'):
    print('mmcv not found, trying openmim install...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'openmim'], check=False)
    subprocess.run([sys.executable, '-m', 'mim', 'install', 'mmcv'], check=False)
else:
    print('mmcv already installed.')

if not _has_mod('mmdet'):
    print('mmdet not found, installing/reinstalling mmdet...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', 'mmdet'], check=False)
else:
    print('mmdet already installed.')

print('\n===== verify =====')
for mod in ['mmengine', 'mmcv', 'mmdet', 'pycocotools', 'timm', 'torchprofile']:
    print(f'{mod}:', _mod_ver(mod))

missing_critical = [mod for mod in ['mmengine', 'mmcv', 'mmdet'] if not _has_mod(mod)]
if missing_critical:
    raise RuntimeError(
        '关键依赖仍缺失: ' + ', '.join(missing_critical) + '\n'
        '这通常不是 notebook 代码问题，而是当前 Colab 的 Python/Torch/CUDA 组合没有可用的 mmcv wheel。\n'
        '优先方案: 切换到兼容运行时后重新执行依赖单元，或改用 Kaggle。'
    )

print('Dependency check finished.')

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch: 2.10.0+cu128
cuda available: True
torch cuda: 12.8
Base packages already installed.
mmcv not found, trying openmim install...
mmdet not found, installing/reinstalling mmdet...

===== verify =====
mmengine: 0.10.7
mmcv: MISSING
mmdet: MISSING
pycocotools: unknown
timm: 1.0.25
torchprofile: 0.1.0


RuntimeError: 关键依赖仍缺失: mmcv, mmdet
这通常不是 notebook 代码问题，而是当前 Colab 的 Python/Torch/CUDA 组合没有可用的 mmcv wheel。
优先方案: 切换到兼容运行时后重新执行依赖单元，或改用 Kaggle。

In [12]:
import importlib
import os
import subprocess
from pathlib import Path

# 启动训练（AMP 混合精度）
for required_mod in ['mmengine', 'mmcv', 'mmdet']:
    try:
        importlib.import_module(required_mod)
    except Exception as e:
        raise RuntimeError(f'缺少关键依赖 {required_mod}: {e}。请先运行上一单元修复依赖。')

cmd = [
    'python', f'{PROJECT_DIR}/tools/train_mmdet3.py',
    CFG_PATH,
    '--work-dir', WORK_DIR,
    '--amp',
]

env = os.environ.copy()
log_path = Path(WORK_DIR) / 'launch_train_stdout_stderr.log'
log_path.parent.mkdir(parents=True, exist_ok=True)

print('Running:', ' '.join(cmd))
print('PYTHONPATH:', env.get('PYTHONPATH', '(not set)'))
print('Log file:', log_path)

with open(log_path, 'w', encoding='utf-8') as f:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
    )

    for line in proc.stdout:
        print(line, end='')
        f.write(line)

    proc.wait()

print('\nReturn code:', proc.returncode)

if proc.returncode != 0:
    print('\n[ERROR] Training failed. Last 80 log lines:')
    try:
        tail_lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()[-80:]
        for ln in tail_lines:
            print(ln)
    except Exception as e:
        print('Failed to read log tail:', e)
else:
    print('[OK] Training finished successfully.')

RuntimeError: 缺少关键依赖 mmcv: No module named 'mmcv'。请先运行上一单元修复依赖。

In [7]:
import os, subprocess

# 断点续训（取消下方注释以启用）
resume_cmd = [
    'python', f'{PROJECT_DIR}/tools/train_mmdet3.py',
    CFG_PATH,
    '--work-dir', WORK_DIR,
    '--amp',
    '--resume',
]
print('Resume cmd:', ' '.join(resume_cmd))
# subprocess.run(resume_cmd, check=False, env=os.environ.copy())


Resume cmd: python /content/drive/MyDrive/dntr_mmdet3_migration/tools/train_mmdet3.py /content/drive/MyDrive/dntr_mmdet3_migration/configs/aitod_DNTR_mask_mmdet3_train.py --work-dir /content/drive/MyDrive/work_dirs/aitod_DNTR_mask_mmdet3_train --amp --resume
